## שאריות ו-χ²: מידת ההתאמה של המודל לנתונים

המודל התאורטי לטווח הזריקה, ‎`R = (v0**2/g) * sin(2*theta)`‎, הוא קו ישר: ‎`y = m*x + b`‎, כאשר ‎`x = sin(2*theta)`‎ ו-‎`y`‎ הוא הטווח הממוצע בכל זווית. השיפוע ‎`m`‎ אמור לצאת קרוב ל-‎`v0**2/g`‎, והחיתוך ‎`b`‎ קרוב ל-0.

בעמוד זה שני הפרמטרים נתונים כמספרים: ‎`m = 41.3`‎ ו-‎`b = 0.4`‎. העמודים הבאים מראים כיצד מוצאים מהנתונים את הפרמטרים הטובים ביותר. כאן נבחנת שאלה שקודמת לכך: עד כמה מודל עם פרמטרים נתונים מתאים למדידות. לשם כך משמשים שני כלים:

1. **שאריות** (residuals): ‎`resid = y - (m*x + b)`‎ -- ההפרש בין המדידה לתחזית המודל, לכל נקודה.
2. **χ²** (חי בריבוע): מספר אחד שמסכם את כל השאריות, כל אחת ביחס לשגיאת המדידה של הנקודה שלה.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())
x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean()
              for a in angles])

m = 41.3   # slope (given)
b = 0.4    # intercept (given)

y_pred = m*x + b
resid = y - y_pred
print("שאריות:", np.round(resid, 3))

### גרף השאריות

גרף של השאריות מול ‎`x`‎, עם קו אופקי ב-0, מראה במבט אחד אם המודל מתאים לנתונים. כאשר הנקודות מפוזרות משני צדי הקו בלי מבנה, המודל מתאר את הנתונים היטב. מבנה שיטתי, למשל קשת או רצף של שאריות באותו סימן, מעיד שצורת המודל או הפרמטרים שלו אינם מתאימים.

In [ ]:
fig, ax = plt.subplots()
ax.axhline(0, color="gray", linewidth=1)
ax.plot(x, resid, "o")
ax.set_xlabel("sin(2θ)")
ax.set_ylabel("Residual [m]")
plt.show()

השאריות קטנות ביחס לטווחים עצמם (20–40 מטר), ואין בהן מבנה שיטתי. שתי השאריות הגדולות ביותר שייכות לזוויות 30 ו-60 מעלות, שבשתיהן $x \approx 0.87$, והן בסימנים הפוכים: לשתי הנקודות אותו ‎`x`‎, ולכן שום קו ישר אינו עובר דרך שתיהן. השאלה אם שאריות בגודל של כ-2 מטר סבירות תלויה בשגיאת המדידה של כל נקודה, ואת זה בדיוק מודד χ².

### חי בריבוע (χ²)

לכל נקודה יש שגיאת מדידה ‎`sigma_y`‎, כאן ה-SEM של הטווח בכל זווית. χ² מחלק כל שארית בשגיאת המדידה של הנקודה שלה, ומסכם את הריבועים:

$$\chi^2 = \sum \left(\frac{y_i - \hat y_i}{\sigma_{y_i}}\right)^2$$

כאשר $\hat y_i$ היא תחזית המודל (‎`y_pred`‎). שארית בגודל שגיאת המדידה תורמת לסכום בערך 1.

In [ ]:
sigma_y = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1)
    / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])
print("sigma_y לכל זווית:", np.round(sigma_y, 3))

chi2 = np.sum((resid / sigma_y)**2)
print(f"chi^2 = {chi2:.3f}")

### חי בריבוע מצומצם

הערך של χ² גדל עם מספר הנקודות ‎`n`‎, ולכן מחלקים אותו במספר **דרגות החופש**. מספר זה נתון: ‎`dof = n - k`‎, כאשר ‎`k = 2`‎ הוא מספר הפרמטרים של המודל (‎`m`‎ ו-‎`b`‎).

$$\chi^2_\nu = \frac{\chi^2}{n - k}$$

**פירוש**: $\chi^2_\nu \approx 1$ -- ההתאמה טובה, השאריות בגודל שמתאים לשגיאות המדידה. $\chi^2_\nu \gg 1$ -- המודל לא מסביר את הנתונים היטב (או ששגיאות המדידה מוערכות בחסר). $\chi^2_\nu \ll 1$ -- שגיאות המדידה מוערכות ביתר (או "התאמת יתר").

In [ ]:
n = len(x)
k = 2
dof = n - k

reduced_chi2 = chi2 / dof
print(f"chi^2_nu (מצומצם) = {reduced_chi2:.3f}   (dof={dof})")

כאן $\chi^2_\nu \approx 1.3$, קרוב ל-1: השאריות בגודל שמתאים לשגיאות המדידה, והמודל עם הפרמטרים הנתונים מתאים לנתונים.

### באג נפוץ: χ² בלי חלוקה בשגיאת המדידה

1. לחשב ‎`np.sum(resid**2)`‎ ולקרוא לזה "chi-squared" -- זה סכום ריבועי שאריות **גולמי**, ללא נרמול לפי שגיאת המדידה של כל נקודה. שתי נקודות עם אותה שארית אבל ‎`sigma_y`‎ שונה לגמרי צריכות לתרום תרומה שונה ל-$\chi^2$ -- בלי החלוקה, ההבדל הזה נעלם.
2. לבלבל בין $\chi^2$ (הסכום הגולמי) לבין $\chi^2_\nu$ (המצומצם, אחרי חלוקה ב-dof) -- $\chi^2$ גדל עם ‎`n`‎ גם כשההתאמה טובה, ולכן **אי אפשר** לפרש אותו לבד כ"טוב/רע" בלי לדעת גם את ‎`n`‎ ו-‎`k`‎.

In [ ]:
chi2_no_norm = np.sum(resid**2)   # no division by sigma_y: not chi-squared
print(f"סכום שאריות בריבוע (לא מנורמל): {chi2_no_norm:.3f}")
print(f"chi^2 אמיתי (מנורמל לפי sigma_y): {chi2:.3f}")
print("שני המספרים שונים: ההבדל ביניהם הוא החלוקה בשגיאת המדידה.")

### נסו בעצמכם

הפרמטרים התאורטיים של אותו מודל, בלי שום מידע מהנתונים, הם ‎`m = v0**2/g`‎ עם ‎`v0 = 20`‎, ו-‎`b = 0`‎. חשבו עבורם את השאריות ‎`resid_th`‎, את ‎`chi2_th`‎ ואת ‎`reduced_chi2_th`‎, והשוו ל-‎`reduced_chi2`‎ של הפרמטרים הנתונים. באילו פרמטרים χ² המצומצם קטן יותר? מה ניתן לומר על הסימנים של ‎`resid_th`‎?

In [ ]:
# m_th = 20.0**2 / g
# b_th = 0.0
# resid_th = ...
# chi2_th = ...
# reduced_chi2_th = ...

`````{admonition} פתרון
:class: dropdown, tip
```python
m_th = 20.0**2 / g
b_th = 0.0
resid_th = y - (m_th*x + b_th)
chi2_th = np.sum((resid_th / sigma_y)**2)
reduced_chi2_th = chi2_th / dof

print("שאריות:", np.round(resid_th, 3))
print(f"chi^2_nu, פרמטרים תאורטיים: {reduced_chi2_th:.3f}")
print(f"chi^2_nu, פרמטרים נתונים:   {reduced_chi2:.3f}")
```
הערך של χ² המצומצם של הפרמטרים התאורטיים (כ-1.74) גדול מזה של הפרמטרים הנתונים (כ-1.28), כלומר הפרמטרים הנתונים מתאימים יותר לנתונים. ארבע מחמש השאריות של ‎`resid_th`‎ חיוביות: הקו התאורטי עובר מתחת לרוב המדידות.
`````

### בדקו את עצמכם

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "בגרף שאריות כל הנקודות נמצאות מעל הקו האופקי ב-0. מה ניתן להסיק?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "תחזית המודל נמוכה מהמדידה בכל הנקודות, ולכן הפרמטרים אינם מתאימים (למשל החיתוך b קטן מדי)",
                "correct": true,
                "feedback": "נכון - שארית חיובית פירושה שהמדידה גבוהה מתחזית המודל."
            },
            {
                "answer": "המודל מתאים היטב, כי כל השאריות באותו סימן",
                "correct": false,
                "feedback": "לא - רצף של שאריות באותו סימן הוא מבנה שיטתי, והוא מעיד על התאמה לא טובה."
            },
            {
                "answer": "שגיאות המדידה גדולות מדי",
                "correct": false,
                "feedback": "לא - גרף השאריות לבדו אינו משתמש בשגיאות המדידה כלל."
            },
            {
                "answer": "אין לכך משמעות, כי הסימן של שארית אינו חשוב",
                "correct": false,
                "feedback": "לא - הסימן מראה אם המדידה מעל תחזית המודל או מתחתיה."
            }
        ]
    },
    {
        "question": "התאמה נתנה χ² מצומצם של 0.3. מה זה עשוי לרמז?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "שההתאמה גרועה מאוד ויש לזרוק אותה",
                "correct": false,
                "feedback": "לא - χ² מצומצם נמוך מ-1 בדרך כלל אינו מעיד על התאמה גרועה."
            },
            {
                "answer": "ששגיאות המדידה (sigma_y) הוערכו ביתר (גדולות מדי) ביחס לפיזור בפועל, או שיש התאמת יתר",
                "correct": true,
                "feedback": "נכון - השאריות קטנות משמעותית מכפי שהשגיאות המוצהרות מרמזות."
            },
            {
                "answer": "שהשאריות גדולות פי 3 משגיאות המדידה",
                "correct": false,
                "feedback": "לא - שאריות גדולות משגיאות המדידה נותנות χ² מצומצם גדול מ-1, לא קטן ממנו."
            },
            {
                "answer": "שמספר דרגות החופש שגוי",
                "correct": false,
                "feedback": "לא - מספר דרגות החופש תלוי רק במספר הנקודות ובמספר הפרמטרים, לא בערך של χ²."
            }
        ]
    },
    {
        "question": "במדידה יש 5 נקודות, והמודל הוא קו ישר עם שני פרמטרים. מהו מספר דרגות החופש?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "5",
                "correct": false,
                "feedback": "לא - מספר הנקודות פחות מספר הפרמטרים."
            },
            {
                "answer": "3",
                "correct": true,
                "feedback": "נכון - n - k = 5 - 2 = 3."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא - 2 הוא מספר הפרמטרים, k."
            },
            {
                "answer": "7",
                "correct": false,
                "feedback": "לא - מחסרים את מספר הפרמטרים, לא מוסיפים אותו."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

``````{admonition} תרגול נוסף
:class: dropdown, note
כתבו פונקציה ‎`chi2_values(x, y, sigma_y, m, b)`‎ שמחזירה את χ² ואת χ² המצומצם (עם ‎`dof = n - 2`‎). הריצו אותה על הפרמטרים הנתונים ועל הפרמטרים התאורטיים, ובדקו שהתוצאות זהות לאלה שחושבו למעלה.

```python
# def chi2_values(x, y, sigma_y, m, b):
#     ...
```

`````{admonition} פתרון
:class: dropdown, tip
```python
def chi2_values(x, y, sigma_y, m, b):
    resid = y - (m*x + b)
    chi2 = np.sum((resid / sigma_y)**2)
    dof = len(x) - 2
    return chi2, chi2 / dof

chi2_a, reduced_a = chi2_values(x, y, sigma_y, m, b)
chi2_b, reduced_b = chi2_values(x, y, sigma_y, 20.0**2 / g, 0.0)
print(f"נתונים:  chi^2 = {chi2_a:.3f}   chi^2_nu = {reduced_a:.3f}")
print(f"תאורטי: chi^2 = {chi2_b:.3f}   chi^2_nu = {reduced_b:.3f}")
```
`````
``````